In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
# =========================
# PATHS
# =========================
# PATHS
# =========================
DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"
OUT_DIR  = "./submissions/"

# =========================
# BEST PARAMS (YOUR TUNED)
# =========================
C_VALUE = 0.645
MIN_DF  = 2
MAX_DF  = 0.878
WORD_NG = 2
CHAR_NG = 5

SVD_COMPONENTS = 400
RANDOM_STATE  = 42

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]
def load_and_prepare(path):
	df = pd.read_csv(path)

	for c in ["article", "title", "source"]:
		df[c] = df[c].fillna("").astype(str)

	df["text"] = (df["title"] + " " + df["article"]).str.lower()

	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

	df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)
	return df
df_dev  = load_and_prepare(DEV_PATH)
df_eval = load_and_prepare(EVAL_PATH)

X_dev  = df_dev[["source", "text"] + NUM_COLS]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[["source", "text"] + NUM_COLS]
def make_baseline():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				ngram_range=(1, WORD_NG),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([("pre", pre), ("clf", clf)])
def make_svd():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				ngram_range=(1, WORD_NG),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
		],
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("svd", TruncatedSVD(
			n_components=SVD_COMPONENTS,
			random_state=RANDOM_STATE
		)),
		("clf", LogisticRegression(
			C=C_VALUE,
			class_weight="balanced",
			max_iter=2000,
			n_jobs=-1
		))
	])
print("Training baseline...")
baseline = make_baseline()
baseline.fit(X_dev, y_dev)

print("Training SVD...")
svd_model = make_svd()
svd_model.fit(X_dev, y_dev)
print("Predicting...")
proba_base = baseline.predict_proba(X_eval)
proba_svd  = svd_model.predict_proba(X_eval)

pred_base = proba_base.argmax(axis=1)
pred_svd  = proba_svd.argmax(axis=1)

disagree = np.mean(pred_base != pred_svd)
print(f"Disagreement baseline vs SVD: {disagree:.4f}")
ALPHA = 0.6  # baseline weight

proba_ens = ALPHA * proba_base + (1 - ALPHA) * proba_svd
pred_ens  = proba_ens.argmax(axis=1)
import os
os.makedirs(OUT_DIR, exist_ok=True)

def save_submission(pred, name):
	path = f"{OUT_DIR}/{name}.csv"
	pd.DataFrame({
		"Id": df_eval["Id"].astype(int),
		"Predicted": pred.astype(int)
	}).to_csv(path, index=False)
	print("Saved:", path)

save_submission(pred_base, "submission_baseline")
save_submission(pred_svd,  "submission_svd")
save_submission(pred_ens,  "submission_ensemble")


Training baseline...
Training SVD...
Predicting...
Disagreement baseline vs SVD: 0.1822
Saved: ./submissions//submission_baseline.csv
Saved: ./submissions//submission_svd.csv
Saved: ./submissions//submission_ensemble.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ============================================================
# CONFIG
# ============================================================
DEV_PATH = "../data/raw/development.csv"

N_SPLITS = 5
RANDOM_STATE = 42

C_VALUE = 0.64491922705094
MIN_DF  = 2
MAX_DF  = 0.8782583211530898
WORD_NG_MAX = 2
CHAR_NG_MAX = 5

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(DEV_PATH)

for col in ["article", "title", "source"]:
	df[col] = df[col].fillna("").astype(str)

df["text"] = (df["title"] + " " + df["article"]).str.lower()

df["n_tokens"]    = df["article"].str.split().str.len()
df["title_len"]   = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

X = df[["source", "text"] + NUM_COLS]
y = df["label"].astype(int)

# ============================================================
# MODEL
# ============================================================
def make_baseline():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([("pre", pre), ("clf", clf)])

# ============================================================
# CROSS VALIDATION
# ============================================================
skf = StratifiedKFold(
	n_splits=N_SPLITS,
	shuffle=True,
	random_state=RANDOM_STATE
)

all_true = []
all_pred = []

for fold, (tr, va) in enumerate(skf.split(X, y), 1):
	print(f"\n===== FOLD {fold} =====")

	model = make_baseline()
	model.fit(X.iloc[tr], y.iloc[tr])

	y_pred = model.predict(X.iloc[va])
	y_true = y.iloc[va].values

	print(classification_report(y_true, y_pred, digits=3))
	print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

	all_true.extend(y_true)
	all_pred.extend(y_pred)

# ============================================================
# GLOBAL METRICS
# ============================================================
print("\n===== GLOBAL BASELINE RESULT =====")
print(classification_report(all_true, all_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(all_true, all_pred))

print("Macro F1:", f1_score(all_true, all_pred, average="macro"))



===== FOLD 1 =====
              precision    recall  f1-score   support

           0      0.807     0.682     0.739      4708
           1      0.750     0.828     0.787      2117
           2      0.824     0.825     0.825      2232
           3      0.568     0.575     0.571      1996
           4      0.822     0.923     0.870      1715
           5      0.542     0.538     0.540      2611
           6      0.567     0.829     0.674       621

    accuracy                          0.716     16000
   macro avg      0.697     0.743     0.715     16000
weighted avg      0.721     0.716     0.715     16000

Confusion Matrix:
 [[3211  171  116  321   48  733  108]
 [  65 1753  100   60   20   68   51]
 [  62  133 1841   74    9   52   61]
 [ 159  113  108 1148  130  263   75]
 [  14   12    2   55 1583   43    6]
 [ 444  142   60  337  131 1405   92]
 [  25   14    6   27    5   29  515]]

===== FOLD 2 =====


KeyboardInterrupt: 

In [3]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

# ============================================================
# LOAD
# ============================================================

DEV_PATH = "../data/raw/development.csv"

df = pd.read_csv(DEV_PATH)

for col in ["article", "title", "source", "timestamp"]:
	df[col] = df[col].fillna("").astype(str)

# keep only valid timestamps
df = df[df["timestamp"] != "0000-00-00 00:00:00"].copy()

df["year"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.year
df = df.dropna(subset=["year"])

df["year"] = df["year"].astype(int)

# text
df["text"] = (df["title"] + " " + df["article"]).str.lower()

X = df[["source", "text"]]
y = df["year"]

print("Years distribution:")
print(y.value_counts().sort_index())

# ============================================================
# MODEL
# ============================================================

model = Pipeline([
	("pre", ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("tfidf", TfidfVectorizer(
				ngram_range=(1, 2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=200_000
			), "text"),
		],
		remainder="drop",
		n_jobs=-1
	)),
	("clf", LogisticRegression(
		max_iter=2000,
		n_jobs=-1,
		multi_class="auto"
	))
])

# ============================================================
# CV
# ============================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_true, all_pred = [], []

for fold, (tr, va) in enumerate(skf.split(X, y), 1):
	print(f"\n===== YEAR PREDICTION | FOLD {fold} =====")

	model.fit(X.iloc[tr], y.iloc[tr])
	pred = model.predict(X.iloc[va])

	print(classification_report(y.iloc[va], pred, digits=3))

	all_true.extend(y.iloc[va])
	all_pred.extend(pred)

print("\n===== GLOBAL YEAR PREDICTION =====")
print(classification_report(all_true, all_pred, digits=3))


Years distribution:
year
2004    11243
2005     1403
2006     9316
2007    22857
2008     7428
Name: count, dtype: int64

===== YEAR PREDICTION | FOLD 1 =====


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

        2004      0.899     0.716     0.798      2249
        2005      1.000     0.039     0.075       281
        2006      0.756     0.372     0.499      1863
        2007      0.613     0.942     0.742      4571
        2008      0.796     0.379     0.513      1486

    accuracy                          0.687     10450
   macro avg      0.813     0.490     0.525     10450
weighted avg      0.736     0.687     0.660     10450


===== YEAR PREDICTION | FOLD 2 =====


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

        2004      0.896     0.718     0.797      2249
        2005      0.500     0.018     0.034       281
        2006      0.792     0.388     0.520      1863
        2007      0.609     0.939     0.739      4571
        2008      0.767     0.351     0.481      1486

    accuracy                          0.685     10450
   macro avg      0.713     0.483     0.514     10450
weighted avg      0.723     0.685     0.657     10450


===== YEAR PREDICTION | FOLD 3 =====


...

              precision    recall  f1-score   support

        2004      0.891     0.718     0.795      2249
        2005      0.688     0.039     0.074       280
        2006      0.770     0.396     0.523      1863
        2007      0.616     0.939     0.744      4572
        2008      0.774     0.366     0.497      1485

    accuracy                          0.689     10449
   macro avg      0.748     0.491     0.527     10449
weighted avg      0.727     0.689     0.663     10449


===== YEAR PREDICTION | FOLD 4 =====


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

        2004      0.901     0.705     0.791      2248
        2005      0.692     0.032     0.061       280
        2006      0.755     0.378     0.504      1864
        2007      0.609     0.933     0.737      4572
        2008      0.753     0.373     0.499      1485

    accuracy                          0.681     10449
   macro avg      0.742     0.484     0.518     10449
weighted avg      0.720     0.681     0.655     10449


===== YEAR PREDICTION | FOLD 5 =====


C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

        2004      0.890     0.715     0.793      2248
        2005      0.857     0.021     0.042       281
        2006      0.784     0.381     0.513      1863
        2007      0.609     0.942     0.740      4571
        2008      0.787     0.353     0.487      1486

    accuracy                          0.684     10449
   macro avg      0.785     0.482     0.515     10449
weighted avg      0.733     0.684     0.656     10449


===== GLOBAL YEAR PREDICTION =====
              precision    recall  f1-score   support

        2004      0.895     0.714     0.795     11243
        2005      0.737     0.030     0.058      1403
        2006      0.771     0.383     0.512      9316
        2007      0.611     0.939     0.740     22857
        2008      0.775     0.364     0.496      7428

    accuracy                          0.685     52247
   macro avg      0.758     0.486     0.520     52247
weighted avg      0.728     0.685     0.6

In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
# ============================================================
# LOAD
# ============================================================

DEV_PATH = "../data/raw/development.csv"

df = pd.read_csv(DEV_PATH)

for col in ["article", "title", "source", "timestamp"]:
	df[col] = df[col].fillna("").astype(str)

# keep only valid timestamps
df = df[df["timestamp"] != "0000-00-00 00:00:00"].copy()

df["timestamp_dt"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df[df["timestamp_dt"].notna()].copy()

# text
df["text"] = (df["title"] + " " + df["article"]).str.lower()
# ============================================================
# TEMPORAL LABELS
# ============================================================

df["year"] = df["timestamp_dt"].dt.year.astype(int)

df["semester"] = np.where(
	df["timestamp_dt"].dt.month <= 6, 1, 2
)

df["quadrimester"] = pd.cut(
	df["timestamp_dt"].dt.month,
	bins=[0, 4, 8, 12],
	labels=[1, 2, 3]
).astype(int)

df["week"] = df["timestamp_dt"].dt.isocalendar().week.astype(int)

df["dayofweek"] = df["timestamp_dt"].dt.weekday.astype(int)

df["hour"] = df["timestamp_dt"].dt.hour

df["hour_bucket"] = pd.cut(
	df["hour"],
	bins=[-1, 5, 11, 17, 23],
	labels=["night", "morning", "afternoon", "evening"]
)
# ============================================================
# MODEL
# ============================================================

def make_model():
	return Pipeline([
		("pre", ColumnTransformer(
			transformers=[
				("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
				("tfidf", TfidfVectorizer(
					ngram_range=(1, 2),
					min_df=3,
					max_df=0.9,
					sublinear_tf=True,
					max_features=200_000
				), "text"),
			],
			remainder="drop",
			n_jobs=-1
		)),
		("clf", LogisticRegression(
			max_iter=2000,
			n_jobs=-1,
			class_weight="balanced"
		))
	])
# ============================================================
# CV EVALUATION
# ============================================================

def run_temporal_prediction(df, target_col, n_splits=5):
	print(f"\n================ {target_col.upper()} PREDICTION ================\n")

	X = df[["source", "text"]]
	y = df[target_col]

	print("Label distribution:")
	print(y.value_counts().sort_index(), "\n")

	skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

	all_true, all_pred = [], []

	for fold, (tr, va) in enumerate(skf.split(X, y), 1):
		print(f"\n===== {target_col.upper()} | FOLD {fold} =====")

		model = make_model()
		model.fit(X.iloc[tr], y.iloc[tr])

		pred = model.predict(X.iloc[va])

		print(classification_report(y.iloc[va], pred, digits=3))

		all_true.extend(y.iloc[va])
		all_pred.extend(pred)

	print(f"\n===== GLOBAL {target_col.upper()} PREDICTION =====")
	print(classification_report(all_true, all_pred, digits=3))
run_temporal_prediction(df, "year")
run_temporal_prediction(df, "semester")
run_temporal_prediction(df, "quadrimester")
run_temporal_prediction(df, "week")
run_temporal_prediction(df, "dayofweek")
run_temporal_prediction(df, "hour_bucket")



================ YEAR PREDICTION ================

Label distribution:
year
2004    11243
2005     1403
2006     9316
2007    22857
2008     7428
Name: count, dtype: int64 


===== YEAR | FOLD 1 =====
              precision    recall  f1-score   support

        2004      0.867     0.722     0.788      2249
        2005      0.174     0.363     0.236       281
        2006      0.537     0.604     0.569      1863
        2007      0.733     0.618     0.671      4571
        2008      0.486     0.668     0.563      1486

    accuracy                          0.638     10450
   macro avg      0.559     0.595     0.565     10450
weighted avg      0.677     0.638     0.651     10450


===== YEAR | FOLD 2 =====
              precision    recall  f1-score   support

        2004      0.865     0.721     0.787      2249
        2005      0.210     0.420     0.280       281
        2006      0.553     0.595     0.573      1863
        2007      0.724     0.638     0.678      4571
        200

KeyboardInterrupt: 

In [ ]:
# ============================================================
# TIME CONFIDENCE SHARPENING — FULL KAGGLE PIPELINE
# ============================================================

import pandas as pd
import numpy as np
import re

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ============================================================
# PATHS
# ============================================================

DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"
SUB_PATH  = "../data/submission_time_sharpened.csv"

# ============================================================
# LOAD DATA
# ============================================================

df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

for df in (df_dev, df_eval):
	for col in ["article", "title", "source", "timestamp"]:
		df[col] = df[col].fillna("").astype(str)

# ============================================================
# TEXT + NUMERIC FEATURES
# ============================================================

def build_text(df):
	return (df["title"] + " " + df["article"]).str.lower()

for df in (df_dev, df_eval):
	df["text"] = build_text(df)
	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
	df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

X_DEV  = df_dev[["source", "text"] + NUM_COLS]
y_DEV  = df_dev["label"].astype(int)
X_EVAL = df_eval[["source", "text"] + NUM_COLS]

# ============================================================
# LABEL MODEL (BEST BASELINE)
# ============================================================

label_model = Pipeline([
	("pre", ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				ngram_range=(1, 2),
				min_df=2,
				max_df=0.9,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=2,
				max_df=0.9,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		n_jobs=-1
	)),
	("clf", LogisticRegression(
		C=0.65,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	))
])

print("Training LABEL model...")
label_model.fit(X_DEV, y_DEV)

# ============================================================
# TIME MODEL (SEMESTER)
# ============================================================

df_time = df_dev[df_dev["timestamp"] != "0000-00-00 00:00:00"].copy()
df_time["timestamp_dt"] = pd.to_datetime(df_time["timestamp"], errors="coerce")
df_time = df_time.dropna(subset=["timestamp_dt"])

df_time["semester"] = df_time["timestamp_dt"].dt.month.le(6).map({True: 1, False: 2})

X_TIME = df_time[["source", "text"]]
y_TIME = df_time["semester"]

time_model = Pipeline([
	("pre", ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("tfidf", TfidfVectorizer(
				ngram_range=(1, 2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=150_000
			), "text"),
		],
		n_jobs=-1
	)),
	("clf", LogisticRegression(
		class_weight="balanced",
		max_iter=1500,
		n_jobs=-1
	))
])

print("Training TIME (semester) model...")
time_model.fit(X_TIME, y_TIME)

# ============================================================
# CONFIDENCE SHARPENING
# ============================================================

def confidence_sharpen(label_proba, time_proba,
					   beta=0.6, alpha_min=0.7, alpha_max=1.3):
	conf = time_proba.max(axis=1)
	alpha = 1 + beta * (conf - 0.5)
	alpha = np.clip(alpha, alpha_min, alpha_max)

	out = label_proba ** alpha[:, None]
	out /= out.sum(axis=1, keepdims=True)
	return out

# ============================================================
# INFERENCE ON EVAL
# ============================================================

print("Predicting LABEL probabilities...")
label_proba = label_model.predict_proba(X_EVAL)

print("Predicting TIME probabilities...")
time_proba = time_model.predict_proba(df_eval[["source", "text"]])

print("Applying confidence sharpening...")
final_proba = confidence_sharpen(label_proba, time_proba)

final_pred = final_proba.argmax(axis=1)

# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Saved submission to:", SUB_PATH)


Training LABEL model...
Training TIME (semester) model...
Predicting LABEL probabilities...
Predicting TIME probabilities...
Applying confidence sharpening...
Saved submission to: ../data/submission_time_sharpened.csv


: 